In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/mock")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

products = pd.read_csv(DATA_DIR / "products.csv")
preferences = pd.read_csv(DATA_DIR / "user_preferences.csv")
c1 = pd.read_csv(DATA_DIR / "c1_product_aspects.csv")
c3 = pd.read_csv(DATA_DIR / "c3_temporal_features.csv")

print("Data loaded successfully.")

Data loaded successfully.


In [17]:
display(preferences)

max_budget = preferences["max_budget"].iloc[0]

print("Maximum budget:", max_budget)

,user_id,aspect,importance,confidence,max_budget
0,U001,battery,5,0.90,170000
1,U001,camera,3,0.70,170000
2,U001,performance,4,0.85,170000
3,U001,display,3,0.80,170000
4,U001,durability,2,0.75,170000
5,U001,price,4,0.90,170000


Maximum budget: 170000


In [18]:
features = c1.merge(
    c3,
    on=["product_id", "aspect"],
    how="inner"
)

print("Integrated product-aspect rows:", features.shape)

display(features.head(10))

Integrated product-aspect rows: (30, 9)


,product_id,aspect,aspect_quality,sentiment_score,c1_confidence,recent_quality,trend_score,temporal_uncertainty,emerging_issue_score
0,P001,battery,0.90,0.84,0.91,0.88,0.04,0.08,0.05
1,P001,camera,0.74,0.68,0.88,0.73,0.01,0.10,0.07
2,P001,performance,0.91,0.86,0.93,0.90,0.03,0.07,0.04
3,P001,display,0.82,0.78,0.89,0.81,0.01,0.09,0.05
4,P001,durability,0.79,0.72,0.84,0.78,0.00,0.12,0.08
5,P001,price,0.70,0.65,0.90,0.69,-0.01,0.10,0.06
6,P002,battery,0.77,0.70,0.85,0.70,-0.06,0.25,0.22
7,P002,camera,0.91,0.87,0.92,0.89,0.02,0.11,0.06
8,P002,performance,0.84,0.80,0.90,0.82,-0.01,0.13,0.09
9,P002,display,0.86,0.81,0.91,0.85,0.01,0.10,0.06


In [19]:
features["nominal_aspect_score"] = (
    features["aspect_quality"] +
    features["recent_quality"]
) / 2

display(
    features[
        [
            "product_id",
            "aspect",
            "aspect_quality",
            "recent_quality",
            "nominal_aspect_score"
        ]
    ].head(10)
)

,product_id,aspect,aspect_quality,recent_quality,nominal_aspect_score
0,P001,battery,0.90,0.88,0.890
1,P001,camera,0.74,0.73,0.735
2,P001,performance,0.91,0.90,0.905
3,P001,display,0.82,0.81,0.815
4,P001,durability,0.79,0.78,0.785
5,P001,price,0.70,0.69,0.695
6,P002,battery,0.77,0.70,0.735
7,P002,camera,0.91,0.89,0.900
8,P002,performance,0.84,0.82,0.830
9,P002,display,0.86,0.85,0.855


In [20]:
pref = preferences[["aspect", "importance", "confidence"]].copy()

pref["importance_weight"] = (
    pref["importance"] /
    pref["importance"].sum()
)

display(pref)

print(
    "Total normalized importance:",
    pref["importance_weight"].sum()
)

,aspect,importance,confidence,importance_weight
0,battery,5,0.90,0.238095
1,camera,3,0.70,0.142857
2,performance,4,0.85,0.190476
3,display,3,0.80,0.142857
4,durability,2,0.75,0.095238
5,price,4,0.90,0.190476


Total normalized importance: 0.9999999999999999


In [21]:
ranking_features = features.merge(
    pref,
    on="aspect",
    how="inner"
)

ranking_features["weighted_contribution"] = (
    ranking_features["nominal_aspect_score"] *
    ranking_features["importance_weight"]
)

display(
    ranking_features[
        [
            "product_id",
            "aspect",
            "nominal_aspect_score",
            "importance",
            "importance_weight",
            "weighted_contribution"
        ]
    ].head(12)
)

,product_id,aspect,nominal_aspect_score,importance,importance_weight,weighted_contribution
0,P001,battery,0.890,5,0.238095,0.211905
1,P001,camera,0.735,3,0.142857,0.105000
2,P001,performance,0.905,4,0.190476,0.172381
3,P001,display,0.815,3,0.142857,0.116429
4,P001,durability,0.785,2,0.095238,0.074762
5,P001,price,0.695,4,0.190476,0.132381
6,P002,battery,0.735,5,0.238095,0.175000
7,P002,camera,0.900,3,0.142857,0.128571
8,P002,performance,0.830,4,0.190476,0.158095
9,P002,display,0.855,3,0.142857,0.122143


In [22]:
product_scores = (
    ranking_features
    .groupby("product_id", as_index=False)
    ["weighted_contribution"]
    .sum()
    .rename(
        columns={
            "weighted_contribution": "personalized_score"
        }
    )
)

product_scores = product_scores.merge(
    products,
    on="product_id",
    how="left"
)

display(product_scores)

,product_id,personalized_score,product_name,category,price
0,P001,0.812857,Phone A,smartphone,165000
1,P002,0.811429,Phone B,smartphone,150000
2,P003,0.818810,Phone C,smartphone,175000
3,P004,0.777619,Phone D,smartphone,140000
4,P005,0.823095,Phone E,smartphone,160000


In [23]:
product_scores["within_budget"] = (
    product_scores["price"] <= max_budget
)

display(
    product_scores[
        [
            "product_id",
            "product_name",
            "price",
            "personalized_score",
            "within_budget"
        ]
    ]
)

,product_id,product_name,price,personalized_score,within_budget
0,P001,Phone A,165000,0.812857,True
1,P002,Phone B,150000,0.811429,True
2,P003,Phone C,175000,0.818810,False
3,P004,Phone D,140000,0.777619,True
4,P005,Phone E,160000,0.823095,True


In [24]:
ranking = (
    product_scores[
        product_scores["within_budget"]
    ]
    .sort_values(
        by="personalized_score",
        ascending=False
    )
    .reset_index(drop=True)
)

ranking["rank"] = np.arange(1, len(ranking) + 1)

ranking = ranking[
    [
        "rank",
        "product_id",
        "product_name",
        "price",
        "personalized_score"
    ]
]

display(ranking)

,rank,product_id,product_name,price,personalized_score
0,1,P005,Phone E,160000,0.823095
1,2,P001,Phone A,165000,0.812857
2,3,P002,Phone B,150000,0.811429
3,4,P004,Phone D,140000,0.777619


In [25]:
over_budget = product_scores[
    ~product_scores["within_budget"]
][
    [
        "product_id",
        "product_name",
        "price",
        "personalized_score"
    ]
]

print("Eligible ranking:")
display(ranking)

print("Excluded because of budget:")
display(over_budget)

Eligible ranking:


,rank,product_id,product_name,price,personalized_score
0,1,P005,Phone E,160000,0.823095
1,2,P001,Phone A,165000,0.812857
2,3,P002,Phone B,150000,0.811429
3,4,P004,Phone D,140000,0.777619


Excluded because of budget:


,product_id,product_name,price,personalized_score
2,P003,Phone C,175000,0.81881


In [26]:
output_path = OUTPUT_DIR / "baseline_ranking.csv"

ranking.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path.resolve())

Saved: E:\J26-DS-349\component4\outputs\baseline_ranking.csv


In [27]:
camera_preferences = preferences.copy()

camera_preferences.loc[
    camera_preferences["aspect"] == "camera",
    "importance"
] = 5

camera_preferences.loc[
    camera_preferences["aspect"] == "battery",
    "importance"
] = 2

camera_pref = camera_preferences[
    ["aspect", "importance", "confidence"]
].copy()

camera_pref["importance_weight"] = (
    camera_pref["importance"] /
    camera_pref["importance"].sum()
)

camera_features = features.merge(
    camera_pref,
    on="aspect",
    how="inner"
)

camera_features["weighted_contribution"] = (
    camera_features["nominal_aspect_score"] *
    camera_features["importance_weight"]
)

camera_scores = (
    camera_features
    .groupby("product_id", as_index=False)
    ["weighted_contribution"]
    .sum()
    .rename(
        columns={
            "weighted_contribution": "personalized_score"
        }
    )
)

camera_scores = camera_scores.merge(
    products,
    on="product_id",
    how="left"
)

camera_scores = camera_scores[
    camera_scores["price"] <= max_budget
]

camera_ranking = (
    camera_scores
    .sort_values(
        "personalized_score",
        ascending=False
    )
    .reset_index(drop=True)
)

camera_ranking["rank"] = (
    np.arange(1, len(camera_ranking) + 1)
)

display(
    camera_ranking[
        [
            "rank",
            "product_name",
            "personalized_score"
        ]
    ]
)

,rank,product_name,personalized_score
0,1,Phone B,0.83175
1,2,Phone E,0.81150
2,3,Phone A,0.79350
3,4,Phone D,0.76475


In [28]:
print(
    "Original top product:",
    ranking.iloc[0]["product_name"]
)

print(
    "Camera-priority top product:",
    camera_ranking.iloc[0]["product_name"]
)

Original top product: Phone E
Camera-priority top product: Phone B


In [29]:
assert (
    ranking["price"] <= max_budget
).all()

print("BUDGET TEST: PASSED")

BUDGET TEST: PASSED


In [30]:
assert ranking["personalized_score"].notna().all()
assert ranking["rank"].is_unique
assert ranking["personalized_score"].is_monotonic_decreasing

print("RANKING SANITY TEST: PASSED")

RANKING SANITY TEST: PASSED
